# 🎯 [WEL03_M01] 야전 급수원 가상 수질검사 항목(pH, 탁도, 잔류염소, 중금속, 대장균) 데이터셋 정의
### Google Colab 실습 환경 (장병 복지·지원 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/WEL03/wel03_m01_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장병 복지·지원 (Level 2) |
| **미션 ID** | WEL03_M01 |
| **미션 주제** | 야전 급수원 가상 수질검사 항목(pH, 탁도, 잔류염소, 중금속, 대장균) 데이터셋 정의 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
전장에서는 급수차, 정수 장비, 우물, 하천 취수원 등 급수원의 수질 이상을 빨리 알아채지 못하면 장병의 건강과 부대 활동에 바로 영향을 주지만, 센서 값은 잡음과 고장이 많아 오경보로 불필요하게 급수를 끊는 일도 생긴다. 러시아-우크라이나 전쟁 사례에서 학습 초점으로 삼은 전장 위생·건강 위험 관리를 반영해, 본 프로젝트는 가상 수질검사 데이터를 분석해 위험 신호를 데이터로 판단하는 과제를 다룬다. 교육용 가상 급수원 8곳의 pH, 탁도, 잔류염소, 중금속, 대장균 측정 자료를 코드로 만들어 스키마 정의, 센서 결측·이상치 정제, 기준치 Baseline 판정, 복합 오염 징후 분류 모델, 신뢰도 지수와 오경보 감축, 상황판 대시보드, 급수 중단 HITL 결재선, 대량 오염 부하 시험, AAR 임계치 재조정, 통합 패키지까지 단계별로 구현한다. 최종 목표는 수질 위험 신호를 근거와 함께 빠르게 제시하되, 실제 음용 판정과 급수 중단은 공식 기준과 의무(예방의학) 담당 확인, 지휘관 결심에 따르도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/wel03_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 결재 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/wel03_data_spec.md`, `docs/wel03_aar_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/wel03_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/wel03_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/wel03_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 야전 급수원 가상 수질검사 항목(pH, 탁도, 잔류염소, 중금속, 대장균) 데이터셋 정의
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

야전 급수원 가상 수질검사 항목(pH, 탁도, 잔류염소, 중금속) 데이터셋을 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **야전 급수원 가상 수질검사 항목(pH, 탁도, 잔류염소, 중금속, 대장균)과 센서·전술 환경 정보의 스키마와 데이터 명세를 정의한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **측정 레코드를 급수원 정보, 수질 5항목, 센서 상태, 전술 파라미터(기상, 위치 오차, 통신 지연) 네 묶음 15개 필드로 정의하고 자료형·단위·허용 범위·형식 규칙을 둔다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/wel03_input_schema.json 및 docs/wel03_data_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 장병 복지·지원 실무 및 러시아-우크라이나 전쟁 전장 환경과의 군사적 부합성

#### 미션 PASS 조건

- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건


- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료


- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""야전 급수원 가상 수질검사 항목(pH, 탁도, 잔류염소, 중금속, 대장균)과 급수원·센서·전술 파라미터를 담은 입력 데이터 스키마를 정의하고, 가상 샘플 20건과 오류 주입 데이터로 검증기를 시험한다."""
import json
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 급수원 가상 수질 센서 데이터 생성기 (실제 부대·위치·검사 결과 없음)
# 가상 급수원 8곳 x 14일 x 1시간 간격. 실제 수질(true_*)·센서 측정값·오염 사건·센서 고장 정답을 함께 만든다.
# 기준치는 교육용 가정값이며, 실제 음용 판정은 공인 검사 결과와 의무(예방의학) 담당 기준에 따른다.
# ---------------------------------------------------------------------------
PARAMS = ["ph", "turbidity", "chlorine", "metal", "ecoli"]
SOURCES = [("W01", "water_truck", 1.0, 2.0), ("W02", "water_truck", 3.5, 1.2), ("W03", "purification_unit", 2.2, 4.1),
           ("W04", "purification_unit", 5.0, 3.3), ("W05", "well", 0.8, 5.6), ("W06", "well", 4.4, 6.0),
           ("W07", "river_intake", 6.3, 1.8), ("W08", "river_intake", 6.9, 4.9)]
BASE = {"water_truck": (7.2, 0.25, 0.75, 1.5), "purification_unit": (7.0, 0.20, 0.85, 1.2),
        "well": (6.8, 0.45, 0.50, 3.0), "river_intake": (7.3, 0.55, 0.60, 2.0)}   # pH, 탁도(NTU), 잔류염소(mg/L), 중금속(ug/L)
LIMITS = {"ph": (5.8, 8.5), "turbidity": (1.0, 2.0), "chlorine": (0.2, 0.1), "metal": (5.0, 10.0), "ecoli": (1, 1)}  # (주의, 위험) 가정 기준
EVENT_TYPES = {  # 오염 사건: 변수별 최대 변화량, 최대치 도달 시간(h)
    "sewage": ({"turbidity": 2.8, "chlorine": -0.5, "ecoli": 30, "ph": -0.3}, 4),
    "metal": ({"metal": 22.0, "ph": -0.5}, 4),
    "disinfect_fail": ({"chlorine": -0.8, "ecoli": 6}, 6),
    "chemical": ({"ph": 1.7, "turbidity": 1.0}, 3),
    "creeping_metal": ({"metal": 13.0}, 18),
}
FAULT_TYPES = ["stuck", "spike", "drift", "dropout"]
NOISE = {"ph": 0.04, "turbidity": 0.06, "chlorine": 0.03, "metal": 0.35}
DIGITS = {"ph": 2, "turbidity": 2, "chlorine": 2, "metal": 1}


def make_dataset(days=14, seed=7, n_events=6, n_faults=10, events=None, faults=None, sources=None):
    """events: [(급수원 번호, 사건 유형, 시작 시각 h, 지속 h, 강도)], faults: [(급수원 번호, 고장 유형, 항목, 시작 h, 지속 h)]"""
    rng = np.random.default_rng(seed)
    sources = sources or SOURCES
    T = days * 24
    tt = np.arange(T)
    hour = tt % 24
    night = (hour < 6) | (hour >= 20)
    rain = np.zeros(T)
    for _ in range(max(1, days // 4)):
        s0, d = int(rng.integers(0, T - 14)), int(rng.integers(6, 14))
        rain[s0:s0 + d] = np.maximum(rain[s0:s0 + d], rng.uniform(0.6, 1.0))
    if events is None:
        events = [(int(rng.integers(len(sources))), str(rng.choice(list(EVENT_TYPES))), int(rng.integers(min(24, T // 4), max(T // 4 + 1, T - 48))),
                   int(rng.integers(10, 30)), 1.0) for _ in range(n_events)]
    if faults is None:
        faults = [(int(rng.integers(len(sources))), str(rng.choice(FAULT_TYPES)), str(rng.choice(["ph", "turbidity", "chlorine", "metal"])),
                   int(rng.integers(12, max(13, T - 30))), int(rng.integers(6, 24))) for _ in range(n_faults)]
    out = []
    for si, (sid, stype, x, y) in enumerate(sources):
        ph0, tb0, cl0, mt0 = BASE[stype]
        tr = {"ph": ph0 + 0.08 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 0.03, T),
              "turbidity": tb0 * (1 + 0.1 * rng.normal(0, 1, T)),
              "chlorine": (cl0 * np.exp(-0.022 * hour) if stype == "water_truck" else cl0 + rng.normal(0, 0.03, T)),
              "metal": mt0 + rng.normal(0, 0.2, T), "ecoli": np.zeros(T)}
        if stype in ("river_intake", "well"):
            tr["turbidity"] = tr["turbidity"] + rain * (0.9 if stype == "river_intake" else 0.45)
        f_all, etype = np.zeros(T), np.array(["none"] * T, dtype=object)
        for (s_i, et, onset, dur, scale) in events:
            if s_i != si:
                continue
            delta, ramp = EVENT_TYPES[et]
            f = np.clip((tt - onset + 1) / ramp, 0, 1) * ((tt >= onset) & (tt < onset + dur + ramp))
            for k, v in delta.items():
                tr[k] = tr[k] + v * f * scale
            etype[f > 0] = et
            f_all = np.maximum(f_all, f)
        tr["chlorine"] = np.clip(tr["chlorine"], 0.02, None)
        tr["turbidity"] = np.clip(tr["turbidity"], 0.05, None)
        tr["metal"] = np.clip(tr["metal"], 0.1, None)
        grade = np.where(f_all >= 0.2, 2, 0)
        mild = (tr["turbidity"] > LIMITS["turbidity"][0]) | (tr["chlorine"] < LIMITS["chlorine"][0]) | (tr["metal"] > LIMITS["metal"][0])
        grade = np.where((grade == 0) & mild, 1, grade)
        ms = {}
        for p in ["ph", "turbidity", "chlorine", "metal"]:
            sd = NOISE[p] * np.where(night, 1.5, 1.0) * (tr[p] if p == "turbidity" else 1)
            ms[p] = np.round(np.clip(tr[p] + rng.normal(0, 1, T) * sd, 0, None), DIGITS[p])
        ms["ecoli"] = (rng.poisson(tr["ecoli"]) + (rng.random(T) < 0.004)).astype(float)
        fault, fparam = np.array(["none"] * T, dtype=object), np.array(["-"] * T, dtype=object)
        for (s_i, ft, p, s0, d) in faults:
            if s_i != si:
                continue
            seg = slice(s0, min(T, s0 + d))
            if ft == "stuck":
                ms[p][seg] = ms[p][s0]
            elif ft == "spike":
                seg = slice(s0, s0 + 1)
                ms[p][s0] = {"ph": 11.8, "turbidity": 38.0, "chlorine": 0.0, "metal": 75.0}[p]
            elif ft == "drift":
                ramp = np.linspace(0, 1, seg.stop - s0)
                ms[p][seg] = np.clip(ms[p][seg] + ramp * {"ph": -1.6, "turbidity": 2.6, "chlorine": -0.75, "metal": 12.0}[p], 0, None).round(DIGITS[p])
            elif ft == "dropout":
                for q in PARAMS:
                    ms[q][seg] = np.nan
                p = "all"
            fault[seg], fparam[seg] = ft, p
        df = pd.DataFrame({"source_id": sid, "source_type": stype, "x_km": x, "y_km": y, "t": tt,
                           "cond": np.where(rain > 0, "storm", np.where(night, "night", "day")),
                           **{p: ms[p] for p in PARAMS}, **{f"true_{p}": tr[p] for p in PARAMS},
                           "event": etype, "fault": fault, "fault_param": fparam, "grade": grade})
        out.append(df)
    return pd.concat(out, ignore_index=True)
# ---------------------------------------------------------------------------

# 1) 스키마 정의: 급수원 정보, 수질 측정값(5항목), 센서 상태, 전술 파라미터(기상·위치오차·통신지연)
SCHEMA = {
    "required": ["record_id", "source", "measured_at", "quality", "sensor", "tactical"],
    "fields": {
        "record_id": {"type": str, "pattern": r"^WQ-W\d{2}-\d{4}$"},
        "measured_at": {"type": str, "pattern": r"^D\+\d{2} \d{2}:00$"},
    },
    "source": {
        "source_id": {"type": str, "pattern": r"^W\d{2}$"},
        "source_type": {"type": str, "enum": ["water_truck", "purification_unit", "well", "river_intake"]},
        "grid_km": {"type": list, "length": 2},
    },
    "quality": {   # 단위와 물리적으로 가능한 범위(기준치가 아니라 측정 가능 범위)
        "ph": {"type": (int, float), "min": 0, "max": 14, "unit": "-"},
        "turbidity_ntu": {"type": (int, float), "min": 0, "max": 1000, "unit": "NTU"},
        "chlorine_mgL": {"type": (int, float), "min": 0, "max": 10, "unit": "mg/L"},
        "metal_ugL": {"type": (int, float), "min": 0, "max": 1000, "unit": "ug/L"},
        "ecoli_cfu": {"type": int, "min": 0, "max": 100000, "unit": "CFU/100mL"},
    },
    "sensor": {
        "status": {"type": str, "enum": ["ok", "calibration_due", "fault", "offline"]},
        "last_calibrated_h": {"type": int, "min": 0, "max": 720},
    },
    "tactical": {
        "weather": {"type": str, "enum": ["clear", "rain", "night", "snow", "dust"]},
        "position_error_m": {"type": (int, float), "min": 0, "max": 500},
        "comm_delay_ms": {"type": (int, float), "min": 0, "max": 10000},
    },
}


def check(name, value, rule):
    """값 하나를 규칙(type, enum, min, max, length, pattern)으로 검사해 오류 목록을 돌려준다."""
    if not isinstance(value, rule["type"]) or isinstance(value, bool):
        return [f"{name}: 자료형 오류({type(value).__name__})"]
    errs = []
    if "enum" in rule and value not in rule["enum"]:
        errs.append(f"{name}: 허용되지 않은 값 '{value}'")
    if "min" in rule and value < rule["min"]:
        errs.append(f"{name}: 최소값 {rule['min']} 미만")
    if "max" in rule and value > rule["max"]:
        errs.append(f"{name}: 최대값 {rule['max']} 초과")
    if "length" in rule and len(value) != rule["length"]:
        errs.append(f"{name}: 길이 {rule['length']} 필요")
    if "pattern" in rule and not re.match(rule["pattern"], value):
        errs.append(f"{name}: 형식 불일치 '{value}'")
    return errs


def validate(rec):
    errs = [f"{k}: 필수 필드 누락" for k in SCHEMA["required"] if k not in rec]
    for k, rule in SCHEMA["fields"].items():
        if k in rec:
            errs += check(k, rec[k], rule)
    for group in ("source", "quality", "sensor", "tactical"):
        for k, rule in SCHEMA[group].items():
            if k not in rec.get(group, {}):
                errs.append(f"{group}.{k}: 필수 필드 누락")
            else:
                errs += check(f"{group}.{k}", rec[group][k], rule)
    return errs


# 2) [교육용가상자료] 가상 센서 데이터에서 샘플 20건을 뽑아 스키마 형식(JSON)으로 변환
data = make_dataset(days=3, seed=1, n_events=1, n_faults=0)
rng = np.random.default_rng(1)
rows = data.sample(20, random_state=1).sort_values(["source_id", "t"])
records = []
for _, r in rows.iterrows():
    records.append({
        "record_id": f"WQ-{r['source_id']}-{int(r['t']):04d}", "measured_at": f"D+{int(r['t']) // 24:02d} {int(r['t']) % 24:02d}:00",
        "source": {"source_id": r["source_id"], "source_type": r["source_type"], "grid_km": [r["x_km"], r["y_km"]]},
        "quality": {"ph": float(r["ph"]), "turbidity_ntu": float(r["turbidity"]), "chlorine_mgL": float(r["chlorine"]),
                    "metal_ugL": float(r["metal"]), "ecoli_cfu": int(r["ecoli"])},
        "sensor": {"status": "ok", "last_calibrated_h": int(rng.integers(1, 200))},
        "tactical": {"weather": {"storm": "rain", "night": "night", "day": "clear"}[r["cond"]],
                     "position_error_m": float(round(rng.uniform(3, 40), 1)), "comm_delay_ms": int(rng.integers(30, 900))}})

ok = sum(1 for r in records if not validate(r))
print(f"정상 샘플 {len(records)}건 중 검증 통과 {ok}건, 오류 {len(records) - ok}건")
parsed = json.loads(json.dumps(records, ensure_ascii=False))       # 직렬화 -> 파싱 왕복 시험
print("JSON 직렬화·파싱 왕복 후 값 일치:", parsed == records)

# 3) 오류 주입 시험: 검증기가 잘못된 데이터를 잡아내는지 확인
bad = json.loads(json.dumps(records[0]))
bad["record_id"] = "WQ-1"                              # 형식 오류
bad["quality"]["ph"] = 15.2                            # 물리 범위 초과
bad["quality"]["ecoli_cfu"] = "검출"                    # 자료형 오류(숫자여야 함)
bad["source"]["source_type"] = "bottle"                # 허용되지 않은 급수원 유형
del bad["tactical"]["comm_delay_ms"]                   # 전술 파라미터 누락
print("\n[오류 주입 결과]")
errs = validate(bad)
for e in errs:
    print(" -", e)
print(f"주입 오류 5종 중 검출 {len(errs)}종 -> {'모두 검출' if len(errs) >= 5 else '누락 있음'}")

# 4) 데이터 명세 요약(명세서 초안)
spec = []
for g in ("fields", "source", "quality", "sensor", "tactical"):
    for k, r in SCHEMA[g].items():
        t = r["type"].__name__ if isinstance(r["type"], type) else "number"
        spec.append((g, k, t, r.get("unit", ""), str({x: y for x, y in r.items() if x not in ("type", "unit")})))
print("\n[데이터 명세 요약]")
print(pd.DataFrame(spec, columns=["그룹", "필드", "자료형", "단위", "규칙"]).to_string(index=False))
done = sum(1 for s in spec if s[2] and s[4] != "{}")
print(f"스키마 필드 정의 완성도 {done}/{len(spec)} ({done / len(spec):.0%}): 모든 필드에 자료형·검증 규칙 정의")
print("\n[가정 판정 기준(교육용)] (주의, 위험):", {k: v for k, v in LIMITS.items()})
print("급수원 유형별 샘플 수:", rows["source_type"].value_counts().to_dict())


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
정상 샘플 20건 중 검증 통과 20건, 오류 0건
JSON 직렬화·파싱 왕복 후 값 일치: True

[오류 주입 결과]
 - record_id: 형식 불일치 'WQ-1'
 - source.source_type: 허용되지 않은 값 'bottle'
 - quality.ph: 최대값 14 초과
 - quality.ecoli_cfu: 자료형 오류(str)
 - tactical.comm_delay_ms: 필수 필드 누락
주입 오류 5종 중 검출 5종 -> 모두 검출

[데이터 명세 요약]
      그룹                필드    자료형        단위                                                                     규칙
  fields         record_id    str                                               {'pattern': '^WQ-W\\d{2}-\\d{4}$'}
  fields       measured_at    str                                            {'pattern': '^D\\+\\d{2} \\d{2}:00$'}
  source         source_id    str                                                         {'pattern': '^W\\d{2}$'}
  source       source_type    str           {'enum': ['water_truck', 'purification_unit', 'well', 'river_intake']}
  source           grid_km   list                                                                    {'length': 2}
 quality                ph number         -                                                  {'min': 0, 'max': 14}
 quality     turbidity_ntu number       NTU                                                {'min': 0, 'max': 1000}
 quality      chlorine_mgL number      mg/L                                                  {'min': 0, 'max': 10}
 quality         metal_ugL number      ug/L                                                {'min': 0, 'max': 1000}
 quality         ecoli_cfu    int CFU/100mL                                              {'min': 0, 'max': 100000}
  sensor            status    str                          {'enum': ['ok', 'calibration_due', 'fault', 'offline']}
  sensor last_calibrated_h    int                                                           {'min': 0, 'max': 720}
tactical           weather    str                             {'enum': ['clear', 'rain', 'night', 'snow', 'dust']}
tactical  position_error_m number                                                           {'min': 0, 'max': 500}
tactical     comm_delay_ms number                                                         {'min': 0, 'max': 10000}
스키마 필드 정의 완성도 15/15 (100%): 모든 필드에 자료형·검증 규칙 정의

[가정 판정 기준(교육용)] (주의, 위험): {'ph': (5.8, 8.5), 'turbidity': (1.0, 2.0), 'chlorine': (0.2, 0.1), 'metal': (5.0, 10.0), 'ecoli': (1, 1)}
급수원 유형별 샘플 수: {'purification_unit': 7, 'river_intake': 7, 'water_truck': 3, 'well': 3}
```
</details>
